# MDF Final Project- Getting and Storing Data

## Step 1) Import Libraries

In [17]:
import json
from io import StringIO
from datetime import datetime, timezone

import boto3
import pandas as pd
from sqlalchemy import create_engine

## Step 2) Configurations

In [18]:
AWS_REGION = "us-east-1"
S3_BUCKET = "world-bank-project-data"

RAW_S3_KEY = "raw/worldbank_raw.json"
CLEAN_S3_KEY = "clean/worldbank_panel.csv"

DB_HOST = "worldbank-db.c8hyu8gkak7d.us-east-1.rds.amazonaws.com"
DB_PORT = 5432
DB_NAME = "postgres"
DB_USER = "postgres"
DB_PASSWORD = "Georgetown2026!"

INDICATORS = {
    "IT.NET.USER.ZS": "internet_users_pct",
    "NY.GDP.PCAP.CD": "gdp_per_capita",
    "SP.DYN.LE00.IN": "life_expectancy",
    "SE.SEC.ENRR": "secondary_enrollment_pct",
}

START_YEAR = 2000
END_YEAR = 2023

s3 = boto3.client("s3", region_name=AWS_REGION)

## Step 3) Query the API and Clean Data

In [19]:
all_raw_data = {}
merged = None

for indicator_code, column_name in INDICATORS.items():
    print(f"Fetching {indicator_code}...")

    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator_code}"
    params = {
        "date": f"{START_YEAR}:{END_YEAR}",
        "format": "json",
        "per_page": 20000
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    records = response.json()[1]

    all_raw_data[indicator_code] = records

    df_temp = pd.DataFrame([
        {
            "country_code": r["countryiso3code"],
            "country_name": r["country"]["value"],
            "year": int(r["date"]),
            column_name: float(r["value"])
        }
        for r in records
        if r.get("value") is not None and r.get("countryiso3code")
    ])

    if merged is None:
        merged = df_temp
    else:
        merged = merged.merge(
            df_temp,
            on=["country_code", "country_name", "year"],
            how="outer"
        )

merged = merged[merged["country_code"].str.len() == 3].copy()
merged = merged.sort_values(["country_code", "year"]).reset_index(drop=True)
merged["ingested_at"] = datetime.now(timezone.utc).isoformat()

print(f"Done. Final dataset has {len(merged):,} rows.")
merged.head()

Fetching IT.NET.USER.ZS...
Fetching NY.GDP.PCAP.CD...
Fetching SP.DYN.LE00.IN...
Fetching SE.SEC.ENRR...
Done. Final dataset has 6,264 rows.


,country_code,country_name,year,internet_users_pct,gdp_per_capita,life_expectancy,secondary_enrollment_pct,ingested_at
0,ABW,Aruba,2000,15.442823,20681.023027,72.939,91.580200,2026-04-16T16:11:51.246302+00:00
1,ABW,Aruba,2001,17.100000,20740.132583,73.044,97.556534,2026-04-16T16:11:51.246302+00:00
2,ABW,Aruba,2002,18.800000,21307.248251,73.135,100.163063,2026-04-16T16:11:51.246302+00:00
3,ABW,Aruba,2003,20.800000,21949.485996,73.236,101.929070,2026-04-16T16:11:51.246302+00:00
4,ABW,Aruba,2004,23.000000,23700.631990,73.223,100.940941,2026-04-16T16:11:51.246302+00:00


## Step 4) Upload the Raw JSON and Cleaned CSV to S3

In [20]:
# raw JSON
raw_json = json.dumps(all_raw_data)
s3.put_object(
    Bucket=S3_BUCKET,
    Key=RAW_S3_KEY,
    Body=raw_json.encode("utf-8"),
    ContentType="application/json"
)

# cleaned CSV
merged_clean = merged.copy()

# convert everything numeric-safe before export
for col in INDICATORS.values():
    merged_clean[col] = pd.to_numeric(merged_clean[col], errors="coerce")

csv_buffer = StringIO()

merged_clean.to_csv(
    csv_buffer,
    index=False,
    na_rep="NULL"   # <-- THIS is the fix
)

s3.put_object(
    Bucket=S3_BUCKET,
    Key=CLEAN_S3_KEY,
    Body=csv_buffer.getvalue().encode("utf-8"),
    ContentType="text/csv"
)

print(f"Uploaded raw file to s3://{S3_BUCKET}/{RAW_S3_KEY}")
print(f"Uploaded cleaned file to s3://{S3_BUCKET}/{CLEAN_S3_KEY}")

Uploaded raw file to s3://world-bank-project-data/raw/worldbank_raw.json
Uploaded cleaned file to s3://world-bank-project-data/clean/worldbank_panel.csv


## Step 5) Connect to Database

In [21]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

merged.to_sql(
    "worldbank_panel",
    engine,          # <-- use engine, NOT conn
    if_exists="replace",
    index=False
)

print("Loaded data into RDS table: worldbank_panel")

Loaded data into RDS table: worldbank_panel


### Step 6) Test Query

In [22]:
query = """
SELECT country_name, year, internet_users_pct, gdp_per_capita
FROM worldbank_panel
WHERE year >= 2015
LIMIT 10
"""

df_test = pd.read_sql(query, engine)
df_test

,country_name,year,internet_users_pct,gdp_per_capita
0,Aruba,2015,88.661227,27458.220154
1,Aruba,2016,93.542454,27441.550214
2,Aruba,2017,97.170000,28440.041688
3,Aruba,2018,NaN,30082.158423
4,Aruba,2019,NaN,30645.890602
5,Aruba,2020,NaN,22759.807175
6,Aruba,2021,NaN,26749.329609
7,Aruba,2022,NaN,30975.998912
8,Aruba,2023,NaN,35718.753119
9,Africa Eastern and Southern,2015,14.300000,1498.875240
